In [ ]:
import pandas as pd

def safe_offset(tz):
    try:
        return int(
            pd.Timestamp('2025-09-01')
            .tz_localize(tz)
            .utcoffset()
            .total_seconds() / 3600
        )
    except:
        return None

def clean_flight_data(df, airports_timezone):
    
    df = df.merge(
        airports_timezone[['iata_code', 'UTC_offset']],
        left_on='Origin',
        right_on='iata_code',
        how='left'
    ).drop(columns=['iata_code'])
    
    df = df.merge(
        airports_timezone[['iata_code', 'UTC_offset']],
        left_on='Dest',
        right_on='iata_code',
        how='left',
        suffixes=('', '_dest')
    )
    
    df = df.rename(columns={'UTC_offset_dest': 'Dest_UTC_offset'})
    df = df.drop(columns=['iata_code'])
    
    df['FlightDate'] = pd.to_datetime(df['FlightDate'])
    
    df['CRSDep_local'] = pd.to_datetime(
        df['FlightDate'].astype(str) + ' ' +
        df['CRSDepTime'].astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M',
        errors='coerce'
    )
    
    df['Dep_local'] = pd.to_datetime(
        df['FlightDate'].astype(str) + ' ' +
        df['DepTime'].astype('Int64').astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M',
        errors='coerce'
    )
    
    df['WheelsOff_local'] = pd.to_datetime(
        df['FlightDate'].astype(str) + ' ' +
        df['WheelsOff'].astype('Int64').astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M',
        errors='coerce'
    )
    
    df['CRSDep_UTC'] = df['CRSDep_local'] - pd.to_timedelta(df['UTC_offset'], unit='h')
    df['Dep_UTC'] = df['Dep_local'] - pd.to_timedelta(df['UTC_offset'], unit='h')
    df['WheelsOff_UTC'] = df['WheelsOff_local'] - pd.to_timedelta(df['UTC_offset'], unit='h')
    
    df['CRSArr_UTC'] = df['CRSDep_UTC'] + pd.to_timedelta(df['CRSElapsedTime'], unit='m')
    df['Arr_UTC'] = df['Dep_UTC'] + pd.to_timedelta(df['ActualElapsedTime'], unit='m')
    df['WheelsOn_UTC'] = df['WheelsOff_UTC'] + pd.to_timedelta(df['AirTime'], unit='m')
    
    df = df[df['CRSElapsedTime'] > 0]
    
    delay_cols = [
        'CarrierDelay', 'WeatherDelay', 'NASDelay',
        'SecurityDelay', 'LateAircraftDelay'
    ]
    df[delay_cols] = df[delay_cols].fillna(0)
    
    df = df.drop_duplicates()
    
    return df

airports_timezone = pd.read_csv("stat628_airplanes/airports_timezone.csv")
airports_timezone['UTC_offset'] = airports_timezone['iana_tz'].apply(safe_offset)

file_map = {
    "stat628_airplanes/airlines_2023_9.csv": "stat628_airplanes/sep_2023_clean.csv",
    "stat628_airplanes/airlines_2023_10.csv": "stat628_airplanes/oct_2023_clean.csv",
    "stat628_airplanes/airlines_2023_11.csv": "stat628_airplanes/nov_2023_clean.csv",
    "stat628_airplanes/airlines_2023_12.csv": "stat628_airplanes/dec_2023_clean.csv",
    "stat628_airplanes/airlines_2024_9.csv": "stat628_airplanes/sep_2024_clean.csv",
    "stat628_airplanes/airlines_2024_10.csv": "stat628_airplanes/oct_2024_clean.csv",
    "stat628_airplanes/airlines_2024_11.csv": "stat628_airplanes/nov_2024_clean.csv",
    "stat628_airplanes/airlines_2024_12.csv": "stat628_airplanes/dec_2024_clean.csv",
    "stat628_airplanes/airlines_2025_9.csv": "stat628_airplanes/sep_2025_clean.csv",
    "stat628_airplanes/airlines_2025_10.csv": "stat628_airplanes/oct_2025_clean.csv",
    "stat628_airplanes/airlines_2025_11.csv": "stat628_airplanes/nov_2025_clean.csv"
}

for input_file, output_file in file_map.items():
    df = pd.read_csv(input_file)
    df_clean = clean_flight_data(df, airports_timezone)
    df_clean.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")